<a href="https://colab.research.google.com/github/lisetperez-cmd/AI4ENG_2025-2_Entrega2_PerezLiset_DelCastilloMonica/blob/main/03_modelo_con_preprocesado_y_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
# 03 - Modelo con preprocesado y SVM
# Competencia AI4Eng 2025
# Integrantes: Liset Pérez-Monica del Castillo
# Programa: Ingeniería Industrial
# ================================================


import pandas as pd
import numpy as np
import sklearn as skl

#Se carga y visualiza el archivo train-csv

In [ ]:

# Lectura del CSV
df = pd.read_csv(
    "train.csv",
    engine="python",
    encoding="latin1",
    on_bad_lines="skip"
)

df.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÃ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,TÃ©cnica o tecnolÃ³gica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,TÃ©cnica o tecnolÃ³gica completa,Si,...,N,No,Si,No,TÃ©cnica o tecnolÃ³gica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÃ,Entre 2.5 millones y menos de 4 millones,MÃ¡s de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


In [ ]:
#Se define funcion para determianr valor promedio entre linite inferior y superior
#E_VALORMATRICULAUNIVERSIDAD
def valor_matricula(valor):
    if pd.isna(valor):
        return np.nan
    elif 'Menos de 500 mil' in valor:
        return 250000  # Asignamos un valor promedio entre 0 y 500 mil
    elif 'Entre 500 mil y menos de 1 millón' in valor:
        return 750000  # Promedio entre 500 mil y 1 millón
    elif 'Entre 1 millón y menos de 2.5 millones' in valor:
        return 1750000  # Promedio entre 1 y 2.5 millones
    elif 'Entre 2.5 millones y menos de 4 millones' in valor:
        return 3250000  # Promedio entre 2.5 y 4 millones
    elif 'Entre 4 millones y menos de 5.5 millones' in valor:
        return 4750000  # Promedio entre 4 y 5.5 millones
    elif 'Entre 5.5 millones y menos de 7 millones' in valor:
        return 6250000  # Promedio entre 5.5 y 7 millones
    elif 'Más de 7 millones' in valor:
        return 7500000  # Asignamos un valor mínimo representativo superior a 7 millones
    elif 'No pagó matrícula' in valor:
        return 0  # Asumimos que no se pagó nada
    else:
        return np.nan  # Para cualquier caso que no coincida

In [ ]:
#Se crea copia de df
#Aplicamos funcion
df_copy = df.copy()
df_copy['E_VALORMATRICULAUNIVERSIDAD'] = df_copy['E_VALORMATRICULAUNIVERSIDAD'].apply(valor_matricula)

media = df_copy ['E_VALORMATRICULAUNIVERSIDAD'].mean()
df_copy['E_VALORMATRICULAUNIVERSIDAD'] = df_copy['E_VALORMATRICULAUNIVERSIDAD'].fillna(media.round(3))

df_copy['E_VALORMATRICULAUNIVERSIDAD'].unique() # Se verifica que ya no queden valores nulos en la columna E_VALORMATRICULAUNIVERSIDAD


array([6250000.   , 3250000.   , 4750000.   , 3182500.307,  250000.   ])

In [ ]:
#Se define funcion para determianr valor numerico
#E_HORASSEMANATRABAJA
def horas_trabajadas(valor):
    try:
        # Caso cuando el valor es una cadena de texto que se puede convertir a número
        if isinstance(valor, str):
            valor = valor.lower().strip()

            # Manejo de casos textuales específicos
            if 'menos de 10' in valor:
                return 5  # Promedio estimado
            elif 'entre 11 y 20' in valor:
                return 15  # Promedio estimado
            elif 'más de 30' in valor:
                return 35  # Promedio estimado

            # Eliminar 'horas', 'hora' y convertir a número
            valor = valor.replace(' horas', '').replace('hora', '').strip()
            return float(valor)  # Convertir el valor a float
        return valor  # Si el valor ya es numérico, retornarlo tal cual
    except Exception as e:
        print(f"Error al convertir valor: {valor}, Error: {e}")
        return 0  # En caso de error, devolver 0

In [ ]:
#E_HORASSEMANATRABAJA
df_copy['E_HORASSEMANATRABAJA'] = df_copy['E_HORASSEMANATRABAJA'].apply(horas_trabajadas)

df_copy['E_HORASSEMANATRABAJA'] = df_copy['E_HORASSEMANATRABAJA'].fillna(0)

df_copy['E_HORASSEMANATRABAJA'].unique() # Se verifica que ya no queden valores nulos en la columna E_HORASSEMANATRABAJA


array([ 5.,  0., 15.])

In [ ]:
#Valor número a cada calor en letra columna F_ESTRATOVIVIENDA
# Función para mapear usando palabra clave 'estrato' y el número
# Primero, revisamos los valores únicos
print(df_copy['F_ESTRATOVIVIENDA'].unique())

# Diccionario para mapear
estrato_dict = {
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6,
    'Sin Estrato': 0
}

# Limpiamos espacios extra
df_copy['F_ESTRATOVIVIENDA'] = df_copy['F_ESTRATOVIVIENDA'].str.strip()

# Mapear usando el diccionario
df_copy['F_ESTRATOVIVIENDA_NUM'] = df_copy['F_ESTRATOVIVIENDA'].map(estrato_dict)

# Si hay valores que no se pudieron mapear, los llenamos con 0
df_copy['F_ESTRATOVIVIENDA_NUM'] = df_copy['F_ESTRATOVIVIENDA_NUM'].fillna(0).astype(int)

# Verificamos
print(df_copy[['F_ESTRATOVIVIENDA', 'F_ESTRATOVIVIENDA_NUM']].head(20))

# Valores no reconocidos pasarán a 0
df_copy['F_ESTRATOVIVIENDA_NUM'] = df_copy['F_ESTRATOVIVIENDA_NUM'].fillna(0).astype(int)



['Estrato 3' 'Estrato 4' 'Estrato 5' 'Estrato 2' 'Estrato 1' nan
 'Estrato 6' 'Sin Estrato']
   F_ESTRATOVIVIENDA  F_ESTRATOVIVIENDA_NUM
0          Estrato 3                      3
1          Estrato 3                      3
2          Estrato 3                      3
3          Estrato 4                      4
4          Estrato 3                      3
5          Estrato 5                      5
6          Estrato 2                      2
7          Estrato 2                      2
8          Estrato 1                      1
9          Estrato 5                      5
10         Estrato 1                      1
11         Estrato 2                      2
12         Estrato 3                      3
13         Estrato 1                      1
14         Estrato 3                      3
15         Estrato 3                      3
16         Estrato 1                      1
17         Estrato 2                      2
18         Estrato 1                      1
19         Estrato 2       

In [ ]:
# Mapeo directo en la misma columna
estrato_dict = {
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6,
    'Sin Estrato': 0
}

df_copy['F_ESTRATOVIVIENDA'] = df_copy['F_ESTRATOVIVIENDA'].map(estrato_dict).fillna(0).astype(int)

# Verificar
df_copy['F_ESTRATOVIVIENDA'].unique()

array([3, 4, 5, 2, 1, 0, 6])

In [ ]:
#1-0 se interpetan como Si-No
#F_TIENEINTERNET
# Función para interpretar correctamente Si/No y valores válidos
def convertir_si_no(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
        if valor == 'si':
            return 1
        elif valor == 'no':
            return 0
    elif valor in [1, 0]:
        return valor
    return np.nan  # cualquier otro valor lo dejamos como NaN

# Aplicamos la función
df_copy['F_TIENEINTERNET'] = df_copy['F_TIENEINTERNET'].apply(convertir_si_no)

# Opcional: reemplazar NaN con 0 si quieres
df_copy['F_TIENEINTERNET'] = df_copy['F_TIENEINTERNET'].fillna(0).astype(int)

# Verificamos resultados
df_copy['F_TIENEINTERNET'].unique()

array([1, 0])

In [ ]:
##Se reemplazan los valores de la columna
#E_PAGOMATRICULAPROPIO a datos numéricos,
#"Si" por 1 y "No" por 0. Además, se reemplazan los valores nulos por 0.

df_copy['E_PAGOMATRICULAPROPIO'] = (
    df_copy['E_PAGOMATRICULAPROPIO']
    .astype(str)                  # Convertir todo a texto
    .str.strip()                  # Eliminar espacios al inicio y fin
    .str.lower()                  # Pasar a minúsculas
    .apply(lambda x: 1 if x == 'si' else 0)  # Si 'si' → 1, todo lo demás → 0
)

# Verificar
print(df_copy['E_PAGOMATRICULAPROPIO'].unique())
print(df_copy['E_PAGOMATRICULAPROPIO'].value_counts())

[0 1]
E_PAGOMATRICULAPROPIO
0    150170
1    117589
Name: count, dtype: int64


In [ ]:
#Educación padre-madre
# Diccionario de mapeo
educacion_dict = {
    'ninguno': 0,
    'sin educación': 0,
    'primaria incompleta': 1,
    'primaria completa': 2,
    'secundaria incompleta': 3,
    'secundaria completa': 4,
    'técnica o tecnológica incompleta': 5,
    'técnica o tecnológica completa': 6,
    'educación profesional incompleta': 7,
    'educación profesional completa': 8,
    'postgrado': 9
}

# Función para mapear valores y manejar texto contaminado
def convertir_educacion(valor):
    if pd.isna(valor):
        return 0
    valor_limpio = str(valor).strip().lower()
    return educacion_dict.get(valor_limpio, 0)  # todo lo que no esté en el dict → 0

# Aplicar a madre
df_copy['F_EDUCACIONMADRE'] = df_copy['F_EDUCACIONMADRE'].apply(convertir_educacion)

# Aplicar a padre
df_copy['F_EDUCACIONPADRE'] = df_copy['F_EDUCACIONPADRE'].apply(convertir_educacion)

# Verificar
print(df_copy['F_EDUCACIONMADRE'].unique())
print(df_copy['F_EDUCACIONPADRE'].unique())

[9 0 2 1]
[0 2 1 9]


In [ ]:
#Rendimiento global
rend_dict = {
    'bajo': 1,
    'medio-bajo': 2,
    'medio-alto': 3,
    'alto': 4,
}

df_copy['RENDIMIENTO_GLOBAL'] = df_copy['RENDIMIENTO_GLOBAL'].replace(rend_dict)

df_copy['RENDIMIENTO_GLOBAL'].unique() # Se verifica que ya no queden valores nulos en la columna RENDIMIENTO_GLOBAL


/tmp/ipython-input-1843150577.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_copy['RENDIMIENTO_GLOBAL'] = df_copy['RENDIMIENTO_GLOBAL'].replace(rend_dict)


array([ 3.,  1.,  4.,  2., nan])

In [ ]:
def to_onehot(x):
    values = np.unique(x)
    r = np.r_[[np.argwhere(i==values)[0][0] for i in x]]
    return np.eye(len(values))[r].astype(int)

def replace_column_with_onehot(d, col):
    assert sum(d[col].isna())==0, "column must have no NaN values"
    values = np.unique(d[col]
                      )
    k = to_onehot(d[col].values)
    r = pd.DataFrame(k, columns=["%s_%s"%(col, values[i]) for i in range(k.shape[1])], index=d.index).join(d)
    del(r[col])
    return r

In [ ]:
col_m = 'F_EDUCACIONMADRE'
df_copy[col_m].fillna('No Aplica', inplace=True)
madre_onehot = replace_column_with_onehot(df_copy[[col_m]], col_m)
df_copy = df_copy.join(madre_onehot)
df_copy = df_copy.drop(col_m, axis=1)

col_p = 'F_EDUCACIONPADRE'
df_copy[col_p].fillna('No Aplica', inplace=True)
padre_onehot = replace_column_with_onehot(df_copy[[col_p]], col_p)
df_copy = df_copy.join(padre_onehot)
df_copy = df_copy.drop(col_p, axis=1)

df_copy

/tmp/ipython-input-678676079.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy[col_m].fillna('No Aplica', inplace=True)
/tmp/ipython-input-678676079.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

,ID,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,...,INDICADOR_4,F_ESTRATOVIVIENDA_NUM,F_EDUCACIONMADRE_0,F_EDUCACIONMADRE_1,F_EDUCACIONMADRE_2,F_EDUCACIONMADRE_9,F_EDUCACIONPADRE_0,F_EDUCACIONPADRE_1,F_EDUCACIONPADRE_2,F_EDUCACIONPADRE_9
0,904256,BOGOTÃ,6250000.000,5.0,3,1,Si,Si,N,0,...,0.267,3,0,0,0,1,1,0,0,0
1,645256,ATLANTICO,3250000.000,0.0,3,0,Si,No,N,0,...,0.264,3,1,0,0,0,1,0,0,0
2,308367,BOGOTÃ,3250000.000,0.0,3,1,Si,No,N,0,...,0.264,3,1,0,0,0,1,0,0,0
3,470353,SANTANDER,4750000.000,0.0,4,1,Si,No,N,0,...,0.190,4,1,0,0,0,1,0,0,0
4,989032,ANTIOQUIA,3250000.000,0.0,3,1,Si,Si,N,0,...,0.294,3,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267754,323390,BOGOTÃ,3250000.000,0.0,3,1,Si,Si,N,0,...,0.276,3,1,0,0,0,0,0,1,0
267755,461594,BOGOTÃ,3182500.307,5.0,3,1,No,No,N,0,...,0.269,3,1,0,0,0,1,0,0,0
267756,384765,ATLANTICO,4750000.000,0.0,4,1,Si,No,N,0,...,0.212,4,1,0,0,0,1,0,0,0
267757,355549,ANTIOQUIA,3250000.000,0.0,2,1,Si,No,N,1,...,0.286,2,1,0,0,0,0,0,1,0


In [ ]:
#Eliminamos columnas
df_copy.drop(['E_PRGM_ACADEMICO', 'PERIODO_ACADEMICO'], axis=1, errors='ignore')
for col in ['E_PRGM_ACADEMICO', 'PERIODO_ACADEMICO']:
    if col in df_copy.columns:
        df_copy = df_copy.drop(col, axis=1)

df_copy

,ID,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4,F_ESTRATOVIVIENDA_NUM
0,904256,BOGOTÃ,6250000.000,5.0,3,1,0,Si,Si,N,0,Si,Si,9,3.0,0.322,0.208,0.310,0.267,3
1,645256,ATLANTICO,3250000.000,0.0,3,0,0,Si,No,N,0,Si,No,0,1.0,0.311,0.215,0.292,0.264,3
2,308367,BOGOTÃ,3250000.000,0.0,3,1,0,Si,No,N,0,No,Si,0,1.0,0.297,0.214,0.305,0.264,3
3,470353,SANTANDER,4750000.000,0.0,4,1,0,Si,No,N,0,Si,Si,0,4.0,0.485,0.172,0.252,0.190,4
4,989032,ANTIOQUIA,3250000.000,0.0,3,1,2,Si,Si,N,0,Si,Si,2,2.0,0.316,0.232,0.285,0.294,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267754,323390,BOGOTÃ,3250000.000,0.0,3,1,2,Si,Si,N,0,Si,Si,0,2.0,0.317,0.247,0.320,0.276,3
267755,461594,BOGOTÃ,3182500.307,5.0,3,1,0,No,No,N,0,Si,Si,0,3.0,0.295,0.232,0.284,0.269,3
267756,384765,ATLANTICO,4750000.000,0.0,4,1,0,Si,No,N,0,Si,Si,0,3.0,0.472,0.186,0.271,0.212,4
267757,355549,ANTIOQUIA,3250000.000,0.0,2,1,2,Si,No,N,1,No,Si,0,3.0,0.205,0.298,0.268,0.286,2


In [ ]:
#E_PRGM_DEPARTAMNETO one hot
col_d = 'E_PRGM_DEPARTAMENTO'
depart_onehot = replace_column_with_onehot(df_copy[[col_d]], col_d)
df_copy = df_copy.join(depart_onehot)
df_copy = df_copy.drop(col_d, axis=1)

df_copy

,ID,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,...,E_PRGM_DEPARTAMENTO_NORTE SANTANDER,E_PRGM_DEPARTAMENTO_PUTUMAYO,E_PRGM_DEPARTAMENTO_QUINDIO,E_PRGM_DEPARTAMENTO_RISARALDA,E_PRGM_DEPARTAMENTO_SAN ANDRES,E_PRGM_DEPARTAMENTO_SANTANDER,E_PRGM_DEPARTAMENTO_SUCRE,E_PRGM_DEPARTAMENTO_TOLIMA,E_PRGM_DEPARTAMENTO_VALLE,E_PRGM_DEPARTAMENTO_VAUPES
0,904256,6250000.000,5.0,3,1,Si,Si,N,0,Si,...,0,0,0,0,0,0,0,0,0,0
1,645256,3250000.000,0.0,3,0,Si,No,N,0,Si,...,0,0,0,0,0,0,0,0,0,0
2,308367,3250000.000,0.0,3,1,Si,No,N,0,No,...,0,0,0,0,0,0,0,0,0,0
3,470353,4750000.000,0.0,4,1,Si,No,N,0,Si,...,0,0,0,0,0,1,0,0,0,0
4,989032,3250000.000,0.0,3,1,Si,Si,N,0,Si,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267754,323390,3250000.000,0.0,3,1,Si,Si,N,0,Si,...,0,0,0,0,0,0,0,0,0,0
267755,461594,3182500.307,5.0,3,1,No,No,N,0,Si,...,0,0,0,0,0,0,0,0,0,0
267756,384765,4750000.000,0.0,4,1,Si,No,N,0,Si,...,0,0,0,0,0,0,0,0,0,0
267757,355549,3250000.000,0.0,2,1,Si,No,N,1,No,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#Separamos las columnas que sometemos a evaluación en X, y la predición real en Y.
X_train = df_copy.drop(['RENDIMIENTO_GLOBAL', 'ID'], axis=1)
y_train = df_copy['RENDIMIENTO_GLOBAL']

In [ ]:
#Importamos los clasificadores desde sklearn
#definimos el clasificador DecisionTreetClassifier y lo entrenamos con nuestros datos

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X_train_encoded = X_train.copy()

for col in X_train_encoded.columns:
    if X_train_encoded[col].dtype == 'object':
        X_train_encoded[col] = le.fit_transform(X_train_encoded[col])

y_train_encoded = le.fit_transform(y_train)

model = DecisionTreeClassifier(max_depth=100)
model.fit(X_train_encoded, y_train_encoded)

DecisionTreeClassifier(max_depth=100)

In [ ]:
#Importamos el accuracy_score de sklearn y calculamos el accuracy con los datos de train.

# Unimos X y y temporalmente para eliminar filas incompletas
data = X_train.copy()
data['y'] = y_train

# Eliminamos filas donde y sea NaN
data = data.dropna(subset=['y'])

# Separamos de nuevo
X_train_clean = data.drop(columns=['y'])
y_train_clean = data['y']

# Copiamos los datos limpios
X_train_encoded = X_train_clean.copy()

# Codificar columnas categóricas
encoders = {}

for col in X_train_encoded.columns:
    if X_train_encoded[col].dtype == 'object':
        encoders[col] = LabelEncoder()
        X_train_encoded[col] = encoders[col].fit_transform(X_train_encoded[col])

# Codificar y_train si es categórico
if y_train_clean.dtype == 'object':
    y_encoder = LabelEncoder()
    y_train_encoded = y_encoder.fit_transform(y_train_clean)
else:
    y_train_encoded = y_train_clean

# Entrenar el modelo
model = DecisionTreeClassifier(max_depth=100)
model.fit(X_train_encoded, y_train_encoded)

# Predecir
y_pred = model.predict(X_train_encoded)

# Precisión
accuracy = accuracy_score(y_train_encoded, y_pred)
print("Precisión:", accuracy)

Precisión: 0.9999887958529717


In [ ]:
#Cargamos y mostramos el archivo test.csv.
df_test = pd.read_csv('test.csv')
df_test

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,550236,20183,TRABAJO SOCIAL,BOLIVAR,Menos de 500 mil,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica completa,Si,No,N,Si,Si,Si,Primaria completa,0.328,0.219,0.317,0.247
1,98545,20203,ADMINISTRACION COMERCIAL Y DE MERCADEO,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Técnica o tecnológica completa,0.227,0.283,0.296,0.324
2,499179,20212,INGENIERIA MECATRONICA,BOGOTÁ,Entre 1 millón y menos de 2.5 millones,0,Estrato 3,Si,Secundaria (Bachillerato) incompleta,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,0.285,0.228,0.294,0.247
3,782980,20195,CONTADURIA PUBLICA,SUCRE,Entre 1 millón y menos de 2.5 millones,Entre 21 y 30 horas,Estrato 1,No,Primaria incompleta,Si,No,N,No,No,No,Primaria incompleta,0.160,0.408,0.217,0.294
4,785185,20212,ADMINISTRACION DE EMPRESAS,ATLANTICO,Entre 2.5 millones y menos de 4 millones,Entre 11 y 20 horas,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,0.209,0.283,0.306,0.286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296781,496981,20195,ADMINISTRACION DE EMPRESAS,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 1,Si,Primaria incompleta,Si,Si,N,Si,Si,Si,Primaria incompleta,0.168,0.410,0.235,0.300
296782,209415,20183,DERECHO,META,Entre 1 millón y menos de 2.5 millones,0,Estrato 4,Si,Educación profesional completa,Si,No,N,No,Si,Si,Educación profesional completa,0.471,0.184,0.264,0.193
296783,239074,20212,DERECHO,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Educación profesional completa,0.292,0.249,0.276,0.256
296784,963852,20195,INGENIERIA AERONAUTICA,ANTIOQUIA,Entre 5.5 millones y menos de 7 millones,Entre 11 y 20 horas,Estrato 3,Si,Educación profesional completa,Si,No,N,No,Si,Si,Educación profesional completa,0.305,0.219,0.310,0.260


In [ ]:
#Creamos una copia del df_test y luego aplicamos la función valor_matricula
# a toda la columna de E_VALORMATRICULAUNIVERSIDAD. Por otro lado, cuando todos los valores estén convertidos en números,
#se reemplazarán los nalores nulos con la media de los valores de las columna.

df_test_copy = df_test.copy()
df_test_copy['E_VALORMATRICULAUNIVERSIDAD'] = df_test_copy['E_VALORMATRICULAUNIVERSIDAD'].apply(valor_matricula)

media = df_test_copy ['E_VALORMATRICULAUNIVERSIDAD'].mean()
df_test_copy['E_VALORMATRICULAUNIVERSIDAD'] = df_test_copy['E_VALORMATRICULAUNIVERSIDAD'].fillna(media.round(3))

df_test_copy['E_VALORMATRICULAUNIVERSIDAD'].unique()
# Se verifica que ya no queden valores nulos en la columna E_VALORMATRICULAUNIVERSIDAD

array([ 250000.   , 3250000.   , 1750000.   ,  750000.   , 7500000.   ,
       4750000.   , 6250000.   ,       0.   , 2817658.121])

In [ ]:
#Se aplica la función horas_trabajadas a toda la columna E_HORASSEMANATRABAJA.
#Por otro lado, los valores nulos se reemplazan por 0.
df_test_copy['E_HORASSEMANATRABAJA'] = df_test_copy['E_HORASSEMANATRABAJA'].apply(horas_trabajadas)

df_test_copy['E_HORASSEMANATRABAJA'] = df_test_copy['E_HORASSEMANATRABAJA'].fillna(0)

df_test_copy['E_HORASSEMANATRABAJA'].unique()
# Se verifica que ya no queden valores nulos en la columna E_HORASSEMANATRABAJA


array([ 5.,  0., 15., 35.])

In [ ]:
#Usando el mismo diccionario para el estrato,
#se le asigna el valor numérico a cada valor en letra de la columna F_ESTRATOVIVIENDA.

df_test_copy['F_ESTRATOVIVIENDA'] = df_test_copy['F_ESTRATOVIVIENDA'].replace(estrato_dict)

df_test_copy['F_ESTRATOVIVIENDA'].unique() # Se verifica que ya no queden valores nulos en la columna F_ESTRATOVIVIENDA

/tmp/ipython-input-4247360664.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test_copy['F_ESTRATOVIVIENDA'] = df_test_copy['F_ESTRATOVIVIENDA'].replace(estrato_dict)


array([ 3.,  2.,  1.,  4., nan,  6.,  5.,  0.])

In [ ]:
#Se reemplazan los valores de la columna F_TIENEINTERNET a datos numéricos, "Si" por 1 y "No" por 0. Además,
#se reemplazan los valores nulos por 0.

df_test_copy['F_TIENEINTERNET'] = df_test_copy['F_TIENEINTERNET'].replace({'Si': 1, 'No': 0})

df_test_copy['F_TIENEINTERNET'] = df_test_copy['F_TIENEINTERNET'].fillna(0)

df_test_copy['F_TIENEINTERNET'].unique() # Se verifica que ya no queden valores nulos en la columna F_TIENEINTERNET

/tmp/ipython-input-475268683.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test_copy['F_TIENEINTERNET'] = df_test_copy['F_TIENEINTERNET'].replace({'Si': 1, 'No': 0})


array([1., 0.])

In [ ]:
#Se reemplazan los valores de la columna E_PAGOMATRICULAPROPIO a datos numéricos,
# "Si" por 1 y "No" por 0. Además, se reemplazan los valores nulos por 0.

df_test_copy['E_PAGOMATRICULAPROPIO'] = df_test_copy['E_PAGOMATRICULAPROPIO'].replace({'Si': 1, 'No': 0})

df_test_copy['E_PAGOMATRICULAPROPIO'] = df_test_copy['E_PAGOMATRICULAPROPIO'].fillna(0)

df_test_copy['E_PAGOMATRICULAPROPIO'].unique()
# Se verifica que ya no queden valores nulos en la columna E_PAGOMATRICULAPROPIO

/tmp/ipython-input-1549903922.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test_copy['E_PAGOMATRICULAPROPIO'] = df_test_copy['E_PAGOMATRICULAPROPIO'].replace({'Si': 1, 'No': 0})


array([1., 0.])

In [ ]:
#Convertimos los valores de F_EDUCACIONMADRE y F_EDUCACIONPADRE
#en one hot usando la función. Al final mostramos el df.
col_m = 'F_EDUCACIONMADRE'
df_test_copy[col_m].fillna('No Aplica', inplace=True)
madre_onehot = replace_column_with_onehot(df_test_copy[[col_m]], col_m)
df_test_copy = df_test_copy.join(madre_onehot)
df_test_copy = df_test_copy.drop(col_m, axis=1)

col_p = 'F_EDUCACIONPADRE'
df_test_copy[col_p].fillna('No Aplica', inplace=True)
padre_onehot = replace_column_with_onehot(df_test_copy[[col_p]], col_p)
df_test_copy = df_test_copy.join(padre_onehot)
df_test_copy = df_test_copy.drop(col_p, axis=1)

df_test_copy

/tmp/ipython-input-2937612869.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test_copy[col_m].fillna('No Aplica', inplace=True)
/tmp/ipython-input-2937612869.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,...,F_EDUCACIONPADRE_Ninguno,F_EDUCACIONPADRE_No Aplica,F_EDUCACIONPADRE_No sabe,F_EDUCACIONPADRE_Postgrado,F_EDUCACIONPADRE_Primaria completa,F_EDUCACIONPADRE_Primaria incompleta,F_EDUCACIONPADRE_Secundaria (Bachillerato) completa,F_EDUCACIONPADRE_Secundaria (Bachillerato) incompleta,F_EDUCACIONPADRE_Técnica o tecnológica completa,F_EDUCACIONPADRE_Técnica o tecnológica incompleta
0,550236,20183,TRABAJO SOCIAL,BOLIVAR,250000.0,5.0,3.0,1.0,Si,No,...,0,0,0,0,0,0,0,0,1,0
1,98545,20203,ADMINISTRACION COMERCIAL Y DE MERCADEO,ANTIOQUIA,3250000.0,0.0,2.0,1.0,Si,No,...,0,0,0,0,0,0,1,0,0,0
2,499179,20212,INGENIERIA MECATRONICA,BOGOTÁ,1750000.0,0.0,3.0,1.0,Si,No,...,0,0,0,0,0,0,0,1,0,0
3,782980,20195,CONTADURIA PUBLICA,SUCRE,1750000.0,0.0,1.0,0.0,Si,No,...,0,0,0,0,0,1,0,0,0,0
4,785185,20212,ADMINISTRACION DE EMPRESAS,ATLANTICO,3250000.0,15.0,2.0,1.0,Si,No,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296781,496981,20195,ADMINISTRACION DE EMPRESAS,BOGOTÁ,3250000.0,35.0,1.0,1.0,Si,Si,...,0,0,0,0,0,1,0,0,0,0
296782,209415,20183,DERECHO,META,1750000.0,0.0,4.0,1.0,Si,No,...,0,0,0,0,0,0,0,0,0,0
296783,239074,20212,DERECHO,BOGOTÁ,3250000.0,35.0,3.0,1.0,Si,No,...,0,0,0,0,0,0,1,0,0,0
296784,963852,20195,INGENIERIA AERONAUTICA,ANTIOQUIA,6250000.0,15.0,3.0,1.0,Si,No,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#Eliminamos las columnas de programa académico y el periodo, al final mostramos el df.

df_test_copy = df_test_copy.drop(['E_PRGM_ACADEMICO'], axis=1)
df_test_copy = df_test_copy.drop(['PERIODO_ACADEMICO'], axis=1)

df_test_copy

,ID,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,...,F_EDUCACIONPADRE_Ninguno,F_EDUCACIONPADRE_No Aplica,F_EDUCACIONPADRE_No sabe,F_EDUCACIONPADRE_Postgrado,F_EDUCACIONPADRE_Primaria completa,F_EDUCACIONPADRE_Primaria incompleta,F_EDUCACIONPADRE_Secundaria (Bachillerato) completa,F_EDUCACIONPADRE_Secundaria (Bachillerato) incompleta,F_EDUCACIONPADRE_Técnica o tecnológica completa,F_EDUCACIONPADRE_Técnica o tecnológica incompleta
0,550236,BOLIVAR,250000.0,5.0,3.0,1.0,Si,No,N,1.0,...,0,0,0,0,0,0,0,0,1,0
1,98545,ANTIOQUIA,3250000.0,0.0,2.0,1.0,Si,No,N,0.0,...,0,0,0,0,0,0,1,0,0,0
2,499179,BOGOTÁ,1750000.0,0.0,3.0,1.0,Si,No,N,0.0,...,0,0,0,0,0,0,0,1,0,0
3,782980,SUCRE,1750000.0,0.0,1.0,0.0,Si,No,N,0.0,...,0,0,0,0,0,1,0,0,0,0
4,785185,ATLANTICO,3250000.0,15.0,2.0,1.0,Si,No,N,0.0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296781,496981,BOGOTÁ,3250000.0,35.0,1.0,1.0,Si,Si,N,1.0,...,0,0,0,0,0,1,0,0,0,0
296782,209415,META,1750000.0,0.0,4.0,1.0,Si,No,N,0.0,...,0,0,0,0,0,0,0,0,0,0
296783,239074,BOGOTÁ,3250000.0,35.0,3.0,1.0,Si,No,N,0.0,...,0,0,0,0,0,0,1,0,0,0
296784,963852,ANTIOQUIA,6250000.0,15.0,3.0,1.0,Si,No,N,0.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#Convertimos la columna E_PRGM_DEPARTAMENTO en one hot.
col_d = 'E_PRGM_DEPARTAMENTO'
depart_onehot = replace_column_with_onehot(df_test_copy[[col_d]], col_d)
df_test_copy = df_test_copy.join(depart_onehot)
df_test_copy = df_test_copy.drop(col_d, axis=1)

df_test_copy

,ID,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,...,E_PRGM_DEPARTAMENTO_NORTE SANTANDER,E_PRGM_DEPARTAMENTO_PUTUMAYO,E_PRGM_DEPARTAMENTO_QUINDIO,E_PRGM_DEPARTAMENTO_RISARALDA,E_PRGM_DEPARTAMENTO_SAN ANDRES,E_PRGM_DEPARTAMENTO_SANTANDER,E_PRGM_DEPARTAMENTO_SUCRE,E_PRGM_DEPARTAMENTO_TOLIMA,E_PRGM_DEPARTAMENTO_VALLE,E_PRGM_DEPARTAMENTO_VAUPES
0,550236,250000.0,5.0,3.0,1.0,Si,No,N,1.0,Si,...,0,0,0,0,0,0,0,0,0,0
1,98545,3250000.0,0.0,2.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
2,499179,1750000.0,0.0,3.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
3,782980,1750000.0,0.0,1.0,0.0,Si,No,N,0.0,No,...,0,0,0,0,0,0,1,0,0,0
4,785185,3250000.0,15.0,2.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296781,496981,3250000.0,35.0,1.0,1.0,Si,Si,N,1.0,Si,...,0,0,0,0,0,0,0,0,0,0
296782,209415,1750000.0,0.0,4.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
296783,239074,3250000.0,35.0,3.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
296784,963852,6250000.0,15.0,3.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#Eliminamos la columna Unnamed: 0 que aparecía en el df.
# Eliminar la columna solo si existe
if 'Unnamed: 0' in df_test_copy.columns:
    df_test_copy = df_test_copy.drop(['Unnamed: 0'], axis=1)

df_test_copy

,ID,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,...,E_PRGM_DEPARTAMENTO_NORTE SANTANDER,E_PRGM_DEPARTAMENTO_PUTUMAYO,E_PRGM_DEPARTAMENTO_QUINDIO,E_PRGM_DEPARTAMENTO_RISARALDA,E_PRGM_DEPARTAMENTO_SAN ANDRES,E_PRGM_DEPARTAMENTO_SANTANDER,E_PRGM_DEPARTAMENTO_SUCRE,E_PRGM_DEPARTAMENTO_TOLIMA,E_PRGM_DEPARTAMENTO_VALLE,E_PRGM_DEPARTAMENTO_VAUPES
0,550236,250000.0,5.0,3.0,1.0,Si,No,N,1.0,Si,...,0,0,0,0,0,0,0,0,0,0
1,98545,3250000.0,0.0,2.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
2,499179,1750000.0,0.0,3.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
3,782980,1750000.0,0.0,1.0,0.0,Si,No,N,0.0,No,...,0,0,0,0,0,0,1,0,0,0
4,785185,3250000.0,15.0,2.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296781,496981,3250000.0,35.0,1.0,1.0,Si,Si,N,1.0,Si,...,0,0,0,0,0,0,0,0,0,0
296782,209415,1750000.0,0.0,4.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
296783,239074,3250000.0,35.0,3.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0
296784,963852,6250000.0,15.0,3.0,1.0,Si,No,N,0.0,Si,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#Dividimos los datos en X y Y,
#pero en X eliminamos la columna ID ya que no aporta información necesaria para las predicciones.
#Separamos las columnas que sometemos a evaluación en X, y la predición real en Y.

# Columnas a eliminar
cols_a_eliminar = ['RENDIMIENTO_GLOBAL', 'ID']

# Eliminamos solo las que existan en el DataFrame
X_test = df_test_copy.drop(columns=[col for col in cols_a_eliminar if col in df_test_copy.columns])

# Si la columna RENDIMIENTO_GLOBAL existe, la asignamos a Y_test_pred, si no, dejamos Y_test_pred vacío o con valor nulo
if 'RENDIMIENTO_GLOBAL' in df_test_copy.columns:
    Y_test_pred = df_test_copy['RENDIMIENTO_GLOBAL']
else:
    Y_test_pred = None  # O bien puedes asignar un valor por defecto, si es necesario.

In [ ]:
Y_test_pred_series = pd.Series(Y_test_pred)

rend_dict_rev = {
    1: 'bajo',
    2: 'medio-bajo',
    3: 'medio-alto',
    4: 'alto',
}

submission = pd.DataFrame({
    'ID': df_test['ID'],
    'RENDIMIENTO_GLOBAL': Y_test_pred_series.replace(rend_dict_rev)
})

submission.to_csv('submission.csv', index=False)